# 32장 · LCEL과 LangGraph의 상태 전달

같은 검색·답변 흐름도 LCEL에서는 함수 사이의 반환값으로, LangGraph에서는 상태 갱신으로 연결할 수 있습니다.

같은 질문과 문서, 검색 함수, k를 고정해 LCEL과 LangGraph를 실행합니다. 최종 답변이 같더라도 중간 상태의 필드를 누가 다음 단계로 전달하는지는 두 구현에서 다릅니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## Runnable과 그래프 노드의 상태 전달 비교

검색을 거치면 documents가 생기고 답 작성 단계는 거기서 첫 문장을 꺼냅니다. LCEL은 앞 함수의 반환값 전체를 다음 함수에 넘깁니다. Graph는 노드의 부분 반환값을 상태에 반영합니다. 두 코드가 공유하는 자료는 로컬 키워드 벡터로 찾으며 모델은 호출하지 않습니다.


### 셀 01 · LCEL 함수의 상태 반환

검색 후에도 question이 남도록 입력 딕셔너리와 새 documents를 합칩니다.


**실행 준비** · 30장의 문서 세 건과 retrieve를 불러옵니다. 아래 준비 셀을 한 번 실행한 뒤 이어지는 예제에서 이번 장의 변경을 확인하세요.

In [ ]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


In [1]:
from langchain_core.runnables import RunnableLambda

def search_change(state, *, k=1):
    return {'documents': retrieve(state['question'], k)}

def answer_change(state):
    docs = state['documents']
    return {'answer': docs[0]['text'] if docs else '제공된 자료에서 확인할 수 없습니다.'}

chain = RunnableLambda(lambda state: {**state, **search_change(state)}) | RunnableLambda(lambda state: {**state, **answer_change(state)})
request = {'question': '대출 기간은?', 'documents': [], 'answer': ''}
left = chain.invoke(request)
print(left['answer'])

도서 대출 기간은 14일입니다.


검색 문장이 그대로 답에 들어갔습니다. 두 번째 함수에 전달되는 값에는 원래 질문과 새 문서 목록이 모두 있습니다.


### LCEL에서 이전 키 보존하기

<p>search_change가 만든 사전에는 documents만 있습니다. 파이프에서 이 사전만 반환하면 오른쪽 함수는 question을 받지 못합니다. 예제는 {**state, **search_change(state)}를 반환해 기존 키를 복사하고 검색 결과만 덮어씁니다.</p>


### 셀 02 · StateGraph의 부분 갱신

LCEL은 기본값 k=1을 씁니다. 그래프의 검색 노드는 이 셀의 graph_k를 읽습니다. 먼저 두 값을 모두 1로 두고 실행하세요.


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    documents: list[dict]
    answer: str

graph_k = 1

def graph_search(state):
    return search_change(state, k=graph_k)

builder = StateGraph(State)
builder.add_node('search', graph_search)
builder.add_node('compose', answer_change)
builder.add_edge(START, 'search')
builder.add_edge('search', 'compose')
builder.add_edge('compose', END)
graph = builder.compile()
right = graph.invoke(request)
print(right['answer'], right == left)
for name, state in [("LCEL", left), ("Graph", right)]:
    print(name, [doc["id"] for doc in state["documents"]])

도서 대출 기간은 14일입니다. True
LCEL ['library-1']
Graph ['library-1']


마지막 두 줄에 모두 library-1 하나가 나옵니다. graph_k를 2로 바꾸고 이 셀만 다시 실행하면 그래프 쪽에 library-2가 하나 더 생깁니다. 첫 문서를 읽어 만든 답은 여전히 14일이지만 상태 전체를 비교한 값은 False가 됩니다.

두 답이 같아도 검색 단계가 넘긴 값을 들여다보면 구현 차이가 보입니다. LCEL 함수에서는 기존 필드를 다음 사전에 직접 담았습니다. 그래프에서는 검색 노드가 documents만 돌려줘도 원래 question이 남습니다. 마지막 답만 검사하면 중간에 빠진 필드를 놓치기 쉽습니다.


#### 동일한 검색 결과를 다음 단계에 전달하는 두 방법

```mermaid
flowchart TD
q["같은 질문"] --> l["LCEL 검색"] --> m["다음 입력을 직접 조립"] --> a["답변"]
q --> g["그래프 검색 노드"] --> s["documents를 상태에 반영"] --> b["답변 노드"]
```

LCEL 쪽은 함수가 필요한 입력 키를 직접 구성합니다. LangGraph 쪽은 노드가 돌려준 부분 갱신을 상태에 반영하고 다음 노드를 호출합니다.

### 노드 반환값과 완료 상태

<p>26장에서 익힌 부분 갱신을 이용해 graph_search는 documents만 넘깁니다. compose가 새 목록을 받은 시점까지 걸린 시간을 재려면 각 함수의 앞뒤에서 perf_counter()를 읽으세요. graph.invoke 전체를 재면 두 노드 사이의 실행 비용도 합쳐집니다.</p>


## stream()으로 노드 갱신 추적

마지막 문장만 비교하면 검색부터 달랐는지 알기 어렵습니다. 각 노드의 변경분을 살펴보고, 검색 ID가 같을 때만 답 작성과 인용 처리를 다음 후보로 봅니다.


### 과제 03 · stream()의 첫 노드 갱신

updates 모드에서 첫 갱신을 받아 검색된 첫 문서의 ID를 표시하세요. 셀 02에서 graph_k를 2로 바꾸었을 때도 첫 ID가 같은지 확인하세요. 두 번째 ID가 추가됐는지는 셀 02의 문서 목록에서 볼 수 있습니다.


In [3]:
# next(graph.stream(...))로 첫 갱신을 받으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>first = next(graph.stream(request, stream_mode=&#x27;updates&#x27;))
print(first[&#x27;search&#x27;][&#x27;documents&#x27;][0][&#x27;id&#x27;])</code></pre>
<p>실행 결과</p><pre><code>library-1</code></pre>
<p>library-1은 답을 쓴 뒤에 만들어진 ID가 아닙니다. search의 반환값에 이미 들어 있습니다. 같은 실험의 소요시간을 기록한다면 어느 함수 전후를 측정했는지도 함께 남기세요.</p>
</details>


#### 같은 답을 얻은 두 실행의 중간값

<mark>stream의 갱신 이벤트에는 해당 노드가 반환한 필드가 보입니다.</mark> 마지막 답변이 같아도 LCEL에서 직접 넘긴 문서와 그래프 상태에 남은 문서가 어떻게 전달됐는지는 다릅니다. 첫 이벤트의 노드 이름과 키를 찾고 최종 상태에서 같은 값을 확인하세요.

선수 개념 · 30장 LangGraph의 검색 노드와 답변 노드

### 첫 스트림 항목의 키와 값

next()를 한 번 호출하면 search가 반환한 변경분까지 읽습니다. 뒤의 compose가 반환할 answer는 이 항목에 들어 있지 않습니다.

1. stream · 모드 = updates
2. search · 갱신한 키 = documents
3. 첫 항목 · 바깥 키 = search
4. 첫 ID 출력 · documents[0]["id"] = library-1


### 과제 04 · 검색 노드의 변경분과 최종 상태 비교

updates로 검색 노드의 반환 키를, invoke로 완료 상태의 키를 출력하세요. question을 읽을 수 있는 위치가 어디인지 비교하세요.


In [ ]:
# request를 같은 graph에 전달해 검색 변경분과 최종 키를 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>change = next(graph.stream(request, stream_mode=&#x27;updates&#x27;))
complete = graph.invoke(request)
print(sorted(change[&#x27;search&#x27;]))
print(sorted(complete))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;documents&#x27;]
[&#x27;answer&#x27;, &#x27;documents&#x27;, &#x27;question&#x27;]</code></pre>
<p>검색 노드가 이번에 쓴 키는 documents 하나입니다. question은 요청에 들어 있었고 answer는 compose가 썼습니다. 검색 노드가 이 세 값을 모두 반환해야 한다고 가정하면 스트림 첫 항목에서 question을 꺼낼 때 KeyError가 납니다.</p>
</details>


## 돌아보기

셀 02의 graph_k를 2로 바꾸고 다시 실행하세요. 두 문서 목록에서 달라진 ID를 적고 답 문장이 같은 이유를 설명하세요. graph_k를 1로 되돌려 두 상태가 다시 같은지 확인하세요.


## 최종 답이 같은 두 구현에서 처음 달라진 단계 찾기

두 구현의 답변이 우연히 같아도 사용한 자료의 개수가 다를 수 있습니다. 동일한 질문으로 LCEL과 그래프를 실행해 검색 계약이 갈라진 곳을 찾습니다.

### 셀 06 · 한쪽의 검색 개수만 바꿔서 결과 대조하기

기존 LCEL은 기본 k=1을 씁니다. 그래프의 graph_k만 잠시 2로 바꾸고 즉시 원상 복구합니다. 문장과 문서 ID를 따로 비교하세요.

In [4]:
saved_k = graph_k
try:
    graph_k = 2
    experimental = graph.invoke(request)
finally:
    graph_k = saved_k
baseline_result = chain.invoke(request)
print(experimental["answer"] == baseline_result["answer"])
print([d["id"] for d in baseline_result["documents"]])
print([d["id"] for d in experimental["documents"]])

True
['library-1']
['library-1', 'library-2']


두 답은 같지만 그래프 쪽에 문서가 한 건 더 있습니다. answer_change가 목록 첫 문서만 읽기 때문에 문자열 비교만으로는 검색 개수 변경을 잡지 못했습니다.

### 과제 07 · 결과를 승인할 때 비교할 항목을 함수로 묶기

두 결과의 답변과 순서가 있는 문서 ID 목록을 비교해 달라진 필드 이름을 반환하세요. 실험 결과를 먼저 넣고 기존 설정으로 다시 실행한 결과도 확인하세요.

In [5]:
# answer와 documents를 독립적으로 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def differing_fields(left_result, right_result):
    differences = []
    if left_result[&quot;answer&quot;] != right_result[&quot;answer&quot;]:
        differences.append(&quot;answer&quot;)
    left_ids = [d[&quot;id&quot;] for d in left_result[&quot;documents&quot;]]
    right_ids = [d[&quot;id&quot;] for d in right_result[&quot;documents&quot;]]
    if left_ids != right_ids:
        differences.append(&quot;documents&quot;)
    return differences

print(differing_fields(baseline_result, experimental))
print(differing_fields(baseline_result, graph.invoke(request)))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;documents&#x27;]
[]</code></pre>
<p>첫 실행은 documents, 복구 후에는 빈 목록입니다. ID를 집합으로 바꾸면 두 구현이 같은 자료를 다른 순서로 넘기는 오류를 놓칠 수 있습니다.</p>
</details>

<mark>마지막 답 한 줄이 같아도 검색 단계의 동등성은 따로 확인해야 합니다.</mark> 첫 문서가 같으면 뒤에 불필요한 자료가 붙어도 현재 답변 함수의 문자열은 같습니다.

33장에서는 같은 검색 함수를 노드 대신 호출 가능한 도구로 제공하고 요청과 응답의 연결을 확인합니다.